# Murata vibration forecasting — training & deployment guide

This notebook documents how to use the scripts in this folder (`to_pass_to_Murata/`) to:

1. **Prepare** sensor CSV data (chronological train/val/test split)
2. **Train** a Transformer model (48-point input → 48-step forecast)
3. **Export** a deployment-ready model bundle (`.pth` + `.metadata.json`)

Target column: **`Acceleration RMS`** (30-minute cadence recommended).

> Run cells top-to-bottom. Edit the **Configuration** cell first.

## Package contents

| File | Purpose |
|------|---------|
| `run_transformer_tuning.py` | **Main entry point** — split → train → rank under `runs/` |
| `train_transformer_sweep.py` | Low-level training (called by the runner) |
| `split_csv_chronological_train_val_test.py` | Chronological train/val/test split |
| `select_best_sweep_run.py` | Rank training runs by validation metric |
| `model_meta.py` | Writes deployment metadata JSON + standard filenames |
| `rename_models_to_standard.py` | Migrate legacy `.pth` + sidecar JSON to standard names |
| `forecast_sweep_common.py` | Shared training utilities |
| `requirements.txt` | Python dependencies |

**Typical client workflow:** configure paths → run training → collect deployment bundle from `runs/<best-run>/`.

## 1. Configuration

Edit the paths below for your sensor. All other cells use these variables.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

# --- Package root (this folder) ---
PKG_DIR = Path(".").resolve()
if not (PKG_DIR / "run_transformer_tuning.py").is_file():
    alt = PKG_DIR / "to_pass_to_Murata"
    if (alt / "run_transformer_tuning.py").is_file():
        PKG_DIR = alt.resolve()

print("Package dir:", PKG_DIR)

# --- Your sensor data ---
# Single-sensor CSV with TIMESTAMP + Acceleration RMS (+ optional SENSOR_CODE / SENSOR_DESC)
SOURCE_CSV = PKG_DIR / "data" / "AHU_2_9_Blower_DE_A_30_min.csv"  # <-- EDIT ME

# Where split files and training outputs will be written
SPLIT_OUT_DIR = PKG_DIR / "data" / "splits" / "AHU_2_9_Blower_DE_A_30_min"  # <-- EDIT ME
OUTPUT_ROOT = PKG_DIR / "outputs_AHU_2_9_Blower_DE_A_30_min"  # <-- EDIT ME (optional override)

# Written from SOURCE_CSV (unique SENSOR_NAME / SENSOR_DESC + SENSOR_CODE)
SENSOR_MAPPING_CSV = PKG_DIR / "data" / "sensor_id_name_mapping.csv"

# Training knobs (defaults match production Murata settings)
EPOCHS = 1              # increase for full training (e.g. 100+)
TARGET_SMOOTHING = 48   # causal MA window on Acceleration RMS
DEVICE = "cpu"          # "auto", "cpu", or "cuda"
# Timestamp window filters (strict defaults can leave val/test with 0 windows on gapty CSVs)
REQUIRE_UNIFORM_TIMESTEP = False   # True = only ~30-min cadence windows
MAX_GAP_SECONDS = 0.0              # max gap inside a window; 0 = disable
RUN_TRAINING = True      # set True to actually launch training in later cells

SOURCE_CSV, SPLIT_OUT_DIR

Package dir: C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata


(WindowsPath('C:/Users/NGYX/Desktop/Murata_NGYX/to_pass_to_Murata/data/AHU_2_9_Blower_DE_A_30_min.csv'),
 WindowsPath('C:/Users/NGYX/Desktop/Murata_NGYX/to_pass_to_Murata/data/splits/AHU_2_9_Blower_DE_A_30_min'))

## 2. Environment setup

Install dependencies once (Python **3.10** recommended):

```bash
cd to_pass_to_Murata
pip install -r requirements.txt

# GPU training (match CUDA 12.8 on Murata server):
pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128

# CPU-only:
pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cpu
```

In [2]:
import os

def run_cmd(cmd: list[str], *, cwd: Path | None = None) -> None:
    """Run a shell command with live notebook output."""
    work = cwd or PKG_DIR
    printable = " ".join(str(c) for c in cmd)
    print(f"\n>> cd {work}\n>> {printable}\n", flush=True)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    run_list = [str(c) for c in cmd]
    exe_name = Path(run_list[0]).name.lower() if run_list else ""
    if exe_name.startswith("python") and "-u" not in run_list:
        run_list.insert(1, "-u")

    process = subprocess.Popen(
        run_list,
        cwd=str(work),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)


# Quick sanity checks
import torch
import pandas as pd

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("pandas:", pd.__version__)
print("Scripts present:", all((PKG_DIR / s).is_file() for s in [
    "run_transformer_tuning.py",
    "model_meta.py",
    "split_csv_chronological_train_val_test.py",
]))

Python: 3.10.2
torch: 2.8.0+cpu | cuda: False
pandas: 2.3.3
Scripts present: True


## 3. Input CSV requirements

Your source CSV should contain at minimum:

| Column | Description |
|--------|-------------|
| `TIMESTAMP` | Reading time (**day-first**, e.g. `1/6/2026 0:25` = 1 June 2026) |
| `Acceleration RMS` | Target / feature column |

Optional (used for deployment metadata and `sensor_id_name_mapping.csv`):

| Column | Description |
|--------|-------------|
| `SENSOR_CODE` or `STN_CODE` | Hex sensor ID (e.g. `91B8`) |
| `SENSOR_NAME` or `SENSOR_DESC` | Human-readable name (e.g. `AHU 2-9 Blower DE A`) |

The mapping file is **generated from your input CSV** (one row per unique sensor name). Multi-sensor exports produce a full registry; single-sensor files produce one row.

Recommended cadence: **30 minutes** between consecutive points.

In [3]:
if SOURCE_CSV.is_file():
    preview = pd.read_csv(SOURCE_CSV, nrows=3)
    display(preview.head())
    print(f"Rows (approx): {sum(1 for _ in open(SOURCE_CSV, encoding='utf-8', errors='replace')) - 1}")
else:
    print(f"Source CSV not found yet: {SOURCE_CSV}")
    print("Place your sensor CSV under data/ and update SOURCE_CSV above.")

,TIMESTAMP,SENSOR,SENSOR_CODE,SENSOR_NAME,DATA1,DATA2,DATA3,DATA4,DATA5,DATA6,DATA7,DATA8,DATA9,DATA10,DATA11,DATA12,DATA13,DATA14,SENSOR_DESC,Acceleration RMS
0,2026-01-21 00:15:44,VIBRATION,91B8,AHU 2-9 Blower DE A,3.35,350,1.29,200,0.98,6650,0.83,6687,0.83,325,0.76,3.86,3.49,25.47,AHU 2-9 Blower DE A,3.86
1,2026-01-21 00:45:44,VIBRATION,91B8,AHU 2-9 Blower DE A,3.35,400,0.91,375,0.76,425,0.76,5525,0.68,5537,0.68,3.59,3.35,25.47,AHU 2-9 Blower DE A,3.59
2,2026-01-21 01:15:44,VIBRATION,91B8,AHU 2-9 Blower DE A,3.35,375,1.44,6887,0.98,6787,0.83,6800,0.83,7062,0.83,3.62,3.93,25.47,AHU 2-9 Blower DE A,3.62


Rows (approx): 4876


## 3b. Build `sensor_id_name_mapping.csv`

Scan **SOURCE_CSV** for unique sensors and write a clean two-column mapping (`SENSOR_CODE`, `SENSOR_NAME`). Training does this automatically too; run this cell to preview the table before training.

In [4]:
from model_meta import build_sensor_mapping_dataframe, write_sensor_mapping_csv

if SOURCE_CSV.is_file():
    mapping_df = build_sensor_mapping_dataframe(SOURCE_CSV)
    write_sensor_mapping_csv(SOURCE_CSV, SENSOR_MAPPING_CSV)
    display(mapping_df)
    print(f"Wrote {len(mapping_df)} sensor(s) -> {SENSOR_MAPPING_CSV}")
else:
    print(f"Source CSV not found: {SOURCE_CSV}")

,SENSOR_CODE,SENSOR_NAME
0,91B8,AHU 2-9 Blower DE A


Wrote 1 sensor(s) -> C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\sensor_id_name_mapping.csv


## 4. Step A — Chronological split (optional)

`run_transformer_tuning.py` can split automatically, but you can also split manually:

- Default ratios: **60% train / 20% val / 20% test** (time-ordered, no shuffle)
- Outputs: `train.csv`, `val.csv`, `test.csv`, `split_manifest.json`

In [5]:
if RUN_TRAINING and SOURCE_CSV.is_file():
    SPLIT_OUT_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd([
        sys.executable, "split_csv_chronological_train_val_test.py",
        "--input", str(SOURCE_CSV),
        "--out-dir", str(SPLIT_OUT_DIR),
        "--train-ratio", "0.6",
        "--val-ratio", "0.2",
        "--test-ratio", "0.2",
    ])
else:
    print("Set RUN_TRAINING=True and provide SOURCE_CSV to run the split.")
    print("Or skip — run_transformer_tuning.py can split for you in Step B.")


>> cd C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata
>> c:\Users\NGYX\AppData\Local\Programs\Python\Python310\python.exe split_csv_chronological_train_val_test.py --input C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\AHU_2_9_Blower_DE_A_30_min.csv --out-dir C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\splits\AHU_2_9_Blower_DE_A_30_min --train-ratio 0.6 --val-ratio 0.2 --test-ratio 0.2

Wrote C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\splits\AHU_2_9_Blower_DE_A_30_min\train.csv (2925 rows)
Wrote C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\splits\AHU_2_9_Blower_DE_A_30_min\val.csv (975 rows)
Wrote C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\splits\AHU_2_9_Blower_DE_A_30_min\test.csv (976 rows)
Wrote C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\splits\AHU_2_9_Blower_DE_A_30_min\split_manifest.json


## 5. Step B — Train Transformer (main pipeline)

This runs the full pipeline:

1. Split source CSV (if train/val/test not provided)
2. **Generate `sensor_id_name_mapping.csv`** from unique sensors in the input CSV
3. Train candidate config(s) with quantile bands (0.05 / 0.5 / 0.95)
4. Rank runs → pick best under **`runs/`**
5. Write **`modelType__sensorID__sensorName.pth`** + **`.metadata.json`** in the best run folder

All outputs stay under **`<output-root>/runs/`** (no separate `best_model/` copy).

### Timestamp filters (important)

If training fails with **`val 0/...` or `No valid experiment completed`**, your CSV has timestamp gaps.
The notebook defaults are lenient for demo data:

- `REQUIRE_UNIFORM_TIMESTEP = False`
- `MAX_GAP_SECONDS = 0` (disable max-gap filter)

For strict 30-min cadence only, set `REQUIRE_UNIFORM_TIMESTEP = True` and `MAX_GAP_SECONDS = 1860`.

### CLI equivalent

```bash
cd to_pass_to_Murata
python run_transformer_tuning.py \
  --split-source-csv data/AHU_2_9_Blower_DE_A_30_min.csv \
  --epochs 100 \
  --device cpu \
  --target-smoothing-window 48 \
  --no-require-uniform-timestep \
  --max-consecutive-timestamp-gap-seconds 0
```

`--device` accepts `auto` (default), `cpu`, or `cuda`.

In [6]:
train_cmd = [
    sys.executable, "run_transformer_tuning.py",
    "--split-source-csv", str(SOURCE_CSV),
    "--sensor-mapping-csv", str(SENSOR_MAPPING_CSV),
    "--epochs", str(EPOCHS),
    "--device", DEVICE,
    "--target-smoothing-window", str(TARGET_SMOOTHING),
    "--output-root", str(OUTPUT_ROOT),
    "--max-consecutive-timestamp-gap-seconds", str(MAX_GAP_SECONDS),
]
if REQUIRE_UNIFORM_TIMESTEP:
    train_cmd.append("--require-uniform-timestep")
else:
    train_cmd.append("--no-require-uniform-timestep")

print("Command preview:")
print(" ".join(train_cmd))

if RUN_TRAINING:
    if not SOURCE_CSV.is_file():
        raise FileNotFoundError(f"Missing source CSV: {SOURCE_CSV}")
    run_cmd(train_cmd)
else:
    print("\nSet RUN_TRAINING=True in the Configuration cell to execute.")

Command preview:
c:\Users\NGYX\AppData\Local\Programs\Python\Python310\python.exe run_transformer_tuning.py --split-source-csv C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\AHU_2_9_Blower_DE_A_30_min.csv --sensor-mapping-csv C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\sensor_id_name_mapping.csv --epochs 1 --device cpu --target-smoothing-window 48 --output-root C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\outputs_AHU_2_9_Blower_DE_A_30_min --max-consecutive-timestamp-gap-seconds 0.0 --no-require-uniform-timestep

>> cd C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata
>> c:\Users\NGYX\AppData\Local\Programs\Python\Python310\python.exe run_transformer_tuning.py --split-source-csv C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\AHU_2_9_Blower_DE_A_30_min.csv --sensor-mapping-csv C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\data\sensor_id_name_mapping.csv --epochs 1 --device cpu --target-smoothing-window 48 --output-root C:\Users\NGYX\Deskto

## 6. Training outputs — where to look

After training, inspect **`runs/`** under your output root:

| Path | Description |
|------|-------------|
| `runs/sweep_run_ranking.csv` | All candidate runs ranked |
| `runs/best_run_selection.json` | Best run pointer |
| `runs/<tag>/best_config.json` | Hyperparameters + metrics |
| `runs/<tag>/best_metrics.json` | Test metrics |
| `runs/<tag>/rolling_window_forecasts/stitched_test_forecast.html` | Interactive test plot |
| `runs/<tag>/transformer_*_best.pth` | Best checkpoint |
| `runs/<tag>/rms_forecast__<ID>__<Name>.pth` | **Deployment model** |
| `runs/<tag>/rms_forecast__<ID>__<Name>.metadata.json` | **Deployment metadata** |

In [7]:
# List deployment bundles (*.pth + *.metadata.json) under output root
roots = [OUTPUT_ROOT]
roots.extend(sorted(PKG_DIR.glob("outputs_transformer_tuning*")))

found_any = False
for root in roots:
    if not root.is_dir():
        continue
    metas = sorted(root.rglob("*.metadata.json"))
    pths = sorted(root.rglob("rms_forecast__*.pth"))
    if metas or pths:
        found_any = True
        print(f"\n=== {root} ===")
        for p in pths:
            print("  model:", p.relative_to(root))
        for m in metas:
            print("  meta: ", m.relative_to(root))

if not found_any:
    print("No deployment bundles found yet. Run training first.")


=== C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\outputs_AHU_2_9_Blower_DE_A_30_min ===
  model: runs\cfg_48_k11_lr5e4_wd0.0001\rms_forecast__91B8__AHU_2-9_Blower_DE_A.pth
  meta:  runs\cfg_48_k11_lr5e4_wd0.0001\rms_forecast__91B8__AHU_2-9_Blower_DE_A.metadata.json


## 7. Deployment bundle naming (batch upload)

For platform batch upload, model + metadata must share the same stem:

| File | Pattern | Example |
|------|---------|---------|
| Model | `modelType__sensorID__sensorName.pth` | `rms_forecast__91B8__AHU_2-9_Blower_DE_A.pth` |
| Metadata | `modelType__sensorID__sensorName.metadata.json` | `rms_forecast__91B8__AHU_2-9_Blower_DE_A.metadata.json` |

- **modelType**: `rms_forecast` for Murata Transformer models
- **sensorID**: hex code, e.g. `91B8`, `88B3`
- **sensorName**: spaces → underscores, e.g. `AHU_2-9_Blower_DE_A`

Training writes both files automatically when `sensorId` + `sensorName` are known (from CSV path or mapping CSV).

In [8]:
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))

from model_meta import (
    deployment_metadata_filename,
    deployment_model_filename,
    discover_deployment_model_pairs,
    parse_deployment_filename,
)

example = deployment_model_filename("rms_forecast", "91B8", "AHU 2-9 Blower DE A")
example_meta = deployment_metadata_filename("rms_forecast", "91B8", "AHU 2-9 Blower DE A")
print("Example model:    ", example)
print("Example metadata: ", example_meta)

parsed = parse_deployment_filename(example)
print("Parsed:", json.dumps(parsed, indent=2))

Example model:     rms_forecast__91B8__AHU_2-9_Blower_DE_A.pth
Example metadata:  rms_forecast__91B8__AHU_2-9_Blower_DE_A.metadata.json
Parsed: {
  "modelType": "rms_forecast",
  "sensorId": "91B8",
  "sensorName": "AHU 2-9 Blower DE A",
  "sensorNameFile": "AHU_2-9_Blower_DE_A",
  "basename": "rms_forecast__91B8__AHU_2-9_Blower_DE_A"
}


### Metadata JSON fields (summary)

Key fields your platform can read:

```json
{
  "modelType": "rms_forecast",
  "sensorId": "91B8",
  "sensorName": "AHU 2-9 Blower DE A",
  "inputLen": 48,
  "predLen": 48,
  "valueColumn": "Acceleration RMS",
  "checkpointFile": "rms_forecast__91B8__AHU_2-9_Blower_DE_A.pth",
  "metrics": { "testRmse": 0.04, "testMape": 0.83 }
}
```

In [9]:
# Show metadata from the first bundle found (if any)
meta_path = None
for root in [OUTPUT_ROOT, *PKG_DIR.glob("outputs_transformer_tuning*"), PKG_DIR / "models"]:
    if root.is_dir():
        hits = sorted(root.rglob("*.metadata.json"))
        if hits:
            meta_path = hits[0]
            break

if meta_path:
    payload = json.loads(meta_path.read_text(encoding="utf-8"))
    print("Sample metadata:", meta_path)
    display({k: payload.get(k) for k in [
        "modelType", "sensorId", "sensorName", "inputLen", "predLen",
        "valueColumn", "checkpointFile", "metrics",
    ]})
else:
    print("No .metadata.json found yet.")

Sample metadata: C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\outputs_AHU_2_9_Blower_DE_A_30_min\runs\cfg_48_k11_lr5e4_wd0.0001\rms_forecast__91B8__AHU_2-9_Blower_DE_A.metadata.json


{'modelType': 'rms_forecast',
 'sensorId': '91B8',
 'sensorName': 'AHU 2-9 Blower DE A',
 'inputLen': 48,
 'predLen': 48,
 'valueColumn': 'Acceleration RMS',
 'checkpointFile': 'rms_forecast__91B8__AHU_2-9_Blower_DE_A.pth',
 'metrics': {'valWindowRmse': 0.101101,
  'valQuantileLoss': 0.031208,
  'baselineRmse': 0.090718,
  'testRmse': 0.108498,
  'testMae': 0.082925,
  'testMape': 1.616874,
  'testR2': 0.672227,
  'testMse': 0.011772,
  'headlineMetric': 'testMape',
  'headlineValue': 1.616874,
  'headlineUnit': 'percent'}}

## 8. Migrate legacy model files

If you have old naming (`AHU_2_9_Blower_DE_A_v4.pth` + `.json`), run:

In [10]:
LEGACY_MODELS_DIR = PKG_DIR / "models"  # folder with old .pth + .json pairs

print("Legacy dir:", LEGACY_MODELS_DIR)
print("\nCLI:")
print(f"  python rename_models_to_standard.py {LEGACY_MODELS_DIR}")

# Uncomment to run migration:
# run_cmd([sys.executable, "rename_models_to_standard.py", str(LEGACY_MODELS_DIR)])

Legacy dir: C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\models

CLI:
  python rename_models_to_standard.py C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\models


## 9. Pair models in a upload folder

Use this helper to verify batch-upload folders contain matched `.pth` + `.metadata.json` pairs:

In [11]:
UPLOAD_DIR = PKG_DIR / "models"  # <-- folder to inspect

if UPLOAD_DIR.is_dir():
    pairs = discover_deployment_model_pairs(UPLOAD_DIR)
    if not pairs:
        print(f"No standard bundles in {UPLOAD_DIR}")
    for sensor_id, info in sorted(pairs.items()):
        print(f"{sensor_id}: {Path(info['modelPath']).name}")
        meta = info.get("metadataPath")
        print(f"         + {Path(meta).name if meta else '(missing metadata)'}")
else:
    print(f"Upload dir not found: {UPLOAD_DIR}")

Upload dir not found: C:\Users\NGYX\Desktop\Murata_NGYX\to_pass_to_Murata\models


## 10. Quick reference

### Minimum client checklist

1. Place sensor CSV under `data/`
2. `pip install -r requirements.txt` (+ torch for your hardware)
3. `python run_transformer_tuning.py --split-source-csv data/<sensor>.csv --epochs 100 --device cuda`
4. Collect from output folder:
   - `rms_forecast__<sensorID>__<sensorName>.pth`
   - `rms_forecast__<sensorID>__<sensorName>.metadata.json`
5. Batch-upload both files to the platform (same stem pairs automatically)

### Model behaviour at inference

- **Input**: last 48 smoothed `Acceleration RMS` points (30-min cadence)
- **Output**: next 48 forecast steps (median + optional quantile bands)
- **Smoothing**: causal trailing MA, window = 48 (must match training)

### Support files to include when handing off

- This notebook
- `requirements.txt`
- `data/sensor_id_name_mapping.csv` (auto-generated from your input CSV)
- Example trained bundle (`.pth` + `.metadata.json`)